In [43]:
#Imports 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [44]:
#Parameters
START_DATE = "2018-01-01"
END_DATE = None
PCA_LOOKBACK = 252
N_COMPONENTS = 5
SIGNAL_LOOKBACK = 5
ENTRY_Z = 2.0
TRANSACTION_COST_BPS = 5
TRADING_DAYS = 252

In [45]:
# Stock Universe 
# 30 stocks across multiple sectors/industries 
TICKERS = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "META",
    "NVDA", "JPM", "BAC", "GS", "MS",
    "XOM", "CVX", "JNJ", "PFE", "UNH",
    "WMT", "COST", "HD", "CAT", "BA",
    "KO", "PEP", "DIS", "NFLX", "CSCO",
    "IBM", "V", "MA", "PG", "MCD"
]

In [46]:
#Download Data 
prices = yf.download(TICKERS, start = START_DATE, end = END_DATE, auto_adjust = 1, progress = 0) ["Close"]
prices = prices.dropna(axis=1)
returns = prices.pct_change().dropna()

In [47]:
#Get the first 252 trading days of Returns 
first_year_window = returns.iloc[0:PCA_LOOKBACK] 
#print("Window Shape: ", first_year_window.shape)
#print("Start Date: ", first_year_window.index[0])
#print("End Date: ", first_year_window.index[-1])

In [48]:
#Standardize the returns against volativity 
scaler = StandardScaler()
standardised_returns = scaler.fit_transform(first_year_window) # .fit get the mean and standard 

standardised_returns= pd.DataFrame(
    standardised_returns, 
    index = first_year_window.index, 
    columns = first_year_window.columns 
)
#standardised_returns.head()
#print(standardised_returns.mean().head())
#print(standardised_returns.std(ddof=0).head())

In [49]:
# Run PCA 
pca = PCA(n_components=N_COMPONENTS)

pca_scores = pca.fit_transform(standardised_returns)

explained_variance = pca.explained_variance_ratio_

# Check how much variance each component explains
for i, variance in enumerate(explained_variance):
    print(f"PC{i+1}: {variance:.2%}")

print(f"\nTotal Explained Variance: {explained_variance.sum():.2%}")


PC1: 50.16%
PC2: 9.85%
PC3: 4.85%
PC4: 3.13%
PC5: 2.89%

Total Explained Variance: 70.88%


In [50]:
# Reconstruct returns using the 5 PCA components

reconstructed_standardised = pca.inverse_transform(pca_scores)

print("PCA Scores Shape:", pca_scores.shape)
print("Reconstructed Shape:", reconstructed_standardised.shape)

reconstructed_standardised = pd.DataFrame(
    reconstructed_standardised,
    index=standardised_returns.index,
    columns=standardised_returns.columns
)

comparison = pd.DataFrame({
    "Actual": standardised_returns["AAPL"],
    "PCA_Reconstructed": reconstructed_standardised["AAPL"]
})

comparison.tail()

PCA Scores Shape: (252, 5)
Reconstructed Shape: (252, 30)


,Actual,PCA_Reconstructed
Date,,
2018-12-27,-0.313372,0.317510
2018-12-28,0.054119,-0.033200
2018-12-31,0.534486,0.627618
2019-01-02,0.087096,0.159304
2019-01-03,-5.200368,-1.870022
